## Imports

In [8]:
import numpy as np
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D , MaxPooling2D , Dense , Flatten
from tensorflow.keras.utils import to_categorical

## Cuda Test

In [7]:
import os
os.environ["TF_DIRECTML_DEVICE_INDEX"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import tensorflow as tf

print("All Physical Devices:", tf.config.list_physical_devices())
print("DML Devices:", tf.config.list_physical_devices('DML'))

import time

# Verify the GPU name
gpu_devices = tf.config.list_physical_devices('GPU')
print(f"DirectML is using: {gpu_devices}")

with tf.device('/GPU:0'):
    a = tf.random.normal([5000, 5000])
    b = tf.random.normal([5000, 5000])
    
    print("Starting GPU computation...")
    start = time.time()
    c = tf.matmul(a, b)
    _ = c.numpy() 
    end = time.time()
    
    print(f"Success! Matrix multiplication took: {end - start:.4f} seconds")

All Physical Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
DML Devices: []
DirectML is using: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Starting GPU computation...
Success! Matrix multiplication took: 0.1096 seconds


## Data preperation

In [9]:
(train_images, train_labels), (test_images, test_labels ) = mnist.load_data()

train_images = (train_images/ 255) - 0.5
test_images = (test_images/ 255) - 0.5

In [10]:
train_images = np.expand_dims(train_images, axis=3)
test_images = np.expand_dims(test_images, axis=3)

num_filter = 8
filter_size = 3
pool_size = 2

## Model Preparing

In [11]:
model = Sequential([
    Conv2D(num_filter,filter_size, input_shape=(28,28,1)),
    MaxPooling2D(pool_size=pool_size),
    Flatten(),
    Dense(10,activation='softmax'),
])

model.compile(
    'adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


# Training

In [13]:
model.fit(
    train_images,
    to_categorical(train_labels),
    epochs=3,
    validation_data = (test_images, to_categorical(test_labels))
)

Epoch 1/3
1875/1875 [==============================] - 6s 3ms/step - loss: 0.3612 - accuracy: 0.8945 - val_loss: 0.2344 - val_accuracy: 0.9317
Epoch 2/3
1875/1875 [==============================] - 4s 2ms/step - loss: 0.2100 - accuracy: 0.9397 - val_loss: 0.1732 - val_accuracy: 0.9492
Epoch 3/3
1875/1875 [==============================] - 4s 2ms/step - loss: 0.1540 - accuracy: 0.9562 - val_loss: 0.1370 - val_accuracy: 0.9613


In [14]:
model.save_weights('cnn.h5')

In [ ]:
model.load_weights('cnn.h5')

## Prediction

In [15]:
predictions = model.predict(test_images[:5])
print(np.argmax(predictions, axis=1))
print(test_labels[:5])

1/1 [==============================] - 0s 44ms/step
[7 2 1 0 4]
[7 2 1 0 4]
